# **Agent with persistence memory**

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
#from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = ChatOpenAI(api_key=api_key,
                    base_url=base_url,
                    model="gpt-5.6-luna",
                    temperature=0.7,
                )

ollama = init_chat_model("qwen3.8:latest", model_provider="ollama", temperature=0)

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
# thread 1 memory --- checkpoint is the file system while thread is the file name. So we can have multiple threads with different memory states.
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football team bring string to the match?\n\nBecause they wanted to tie the game! 🏈⚽',
 'explanation': 'The joke is a pun on the phrase **“tie the game.”** In sports, tying a game means ending with the same score as the opposing team. But “tie” can also mean fastening something with **string**, so the team brought string literally to “tie” the game.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football player break up with his girlfriend?\n\nShe said he was too "touchy," always wanted to "score" with other people, and honestly, she was tired of being "on the bench."\n\nHe said, "That\'s fine, babe. I\'ll just go watch the game."\n\nShe said, "You\'re watching *another* one?"\n\nHe said, "Yeah. It\'s the second half."\n\nShe said, "…That\'s the most boring thing I\'ve ever heard."\n\nHe said, "Yeah, that\'s basically what happens."', 'explanation': '# Explanation of the Joke\n\nThe humor here is built on **football (soccer) terminology being used as double entendres** in a relationship argument, and then a final line that undercuts the whole setup.\n\n## The Setup: Football Puns as Relationship Complaints\n\nEach of the girlfriend\'s complaints works on two levels:\n\n- **"Too touchy"** – In football, players are constantly touching the ball, tackling, etc. In a relationship, "touchy" means oversensitive or easil

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football player break up with his girlfriend?\n\nShe said he was too "touchy," always wanted to "score" with other people, and honestly, she was tired of being "on the bench."\n\nHe said, "That\'s fine, babe. I\'ll just go watch the game."\n\nShe said, "You\'re watching *another* one?"\n\nHe said, "Yeah. It\'s the second half."\n\nShe said, "…That\'s the most boring thing I\'ve ever heard."\n\nHe said, "Yeah, that\'s basically what happens."', 'explanation': '# Explanation of the Joke\n\nThe humor here is built on **football (soccer) terminology being used as double entendres** in a relationship argument, and then a final line that undercuts the whole setup.\n\n## The Setup: Football Puns as Relationship Complaints\n\nEach of the girlfriend\'s complaints works on two levels:\n\n- **"Too touchy"** – In football, players are constantly touching the ball, tackling, etc. In a relationship, "touchy" means oversensitive or easi

In [10]:
# thread 2 memory
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to reach the next level of batting! 🏏',
 'explanation': 'The joke is a pun on the phrase **“reach the next level.”**  \n\nThe cricket team brings a **ladder** because a ladder helps you physically reach a higher level. In cricket, **batting** can also improve or advance to a “next level,” so the joke combines the literal use of a ladder with the figurative idea of becoming better at batting. 🏏'}

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to reach the next level of batting! 🏏', 'explanation': 'The joke is a pun on the phrase **“reach the next level.”**  \n\nThe cricket team brings a **ladder** because a ladder helps you physically reach a higher level. In cricket, **batting** can also improve or advance to a “next level,” so the joke combines the literal use of a ladder with the figurative idea of becoming better at batting. 🏏'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1544-2e15-65a0-8002-510abf76cda9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-06T07:05:06.867344+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1544-104c-6938-8001-bb7774ff8f3a'}}, tasks=(), interrupts=())

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to reach the next level of batting! 🏏', 'explanation': 'The joke is a pun on the phrase **“reach the next level.”**  \n\nThe cricket team brings a **ladder** because a ladder helps you physically reach a higher level. In cricket, **batting** can also improve or advance to a “next level,” so the joke combines the literal use of a ladder with the figurative idea of becoming better at batting. 🏏'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1544-2e15-65a0-8002-510abf76cda9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-06T07:05:06.867344+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1544-104c-6938-8001-bb7774ff8f3a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring

### Time Travel

In [13]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1c1544-2e15-65a0-8002-510abf76cda9"}})

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to reach the next level of batting! 🏏', 'explanation': 'The joke is a pun on the phrase **“reach the next level.”**  \n\nThe cricket team brings a **ladder** because a ladder helps you physically reach a higher level. In cricket, **batting** can also improve or advance to a “next level,” so the joke combines the literal use of a ladder with the figurative idea of becoming better at batting. 🏏'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1c1544-2e15-65a0-8002-510abf76cda9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-06T07:05:06.867344+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1544-104c-6938-8001-bb7774ff8f3a'}}, tasks=(), interrupts=())

In [14]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1c1542-cd5c-6bfc-8000-bd45e4414472"}})

StateSnapshot(values={'topic': 'Cricket'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1c1542-cd5c-6bfc-8000-bd45e4414472'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-06T07:04:29.881855+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1542-cd5a-64c8-bfff-79796d40b466'}}, tasks=(PregelTask(id='4aa037e6-7367-efa4-77be-af5ad01dcea9', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to reach the next level of batting! 🏏'}),), interrupts=())

In [15]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1c1542-cd5a-64c8-bfff-79796d40b466"}})

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to climb the league table! 🏏',
 'explanation': 'The joke is a pun. A **ladder** is something you physically climb, while a **league table** shows the teams’ rankings. The cricket team brought a ladder because they wanted to “climb” higher in the league standings. 🏏'}

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to climb the league table! 🏏', 'explanation': 'The joke is a pun. A **ladder** is something you physically climb, while a **league table** shows the teams’ rankings. The cricket team brought a ladder because they wanted to “climb” higher in the league standings. 🏏'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1551-4085-616e-8003-ffa0b8aeda03'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-06T07:10:57.766641+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c154f-f55d-627a-8002-c0a6d44bd492'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they wanted to climb the league table! 🏏'}, next=('generate_explanation',), config={'configurabl

#### Updating State

In [22]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}

In [23]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-06-01T08:53:08.048692+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, tasks=(PregelTask(id='872c0b90-014c-8496-9675-b150967bd288', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stum

### Fault Tolerance

In [40]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [41]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [43]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))